# 2D One-Population Network

This notebook extends the model to a **2D recurrent neural field** with a single population.

## Idea

Neurons are arranged on a **2D grid** with **Mexican-hat connectivity**:
- local excitation  
- broader surround inhibition  

This creates spatial patterns similar to cortical activity.


In [ ]:
import numpy as np
import scipy as scipy
from scipy.linalg import circulant
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cdist
from scipy import signal
from pandas import Series
from random import gauss
import ipywidgets as widgets
from ipywidgets import interact, fixed
import random as rnd
import matplotlib as mpl
from matplotlib.widgets import Slider
from numpy import fft
import scipy.linalg as la          
from scipy import signal
from scipy.stats import norm
import os
from src.2D1PopNeuronModel import compute_firerate, gaussian, recurrent_connections_2d, compute_influence_2d, compute_full_influence_matrix
from src.2D1PopNeuronModel import compute_angular_variance_2d, plot_radial_mean, center_kernel, uncenter_field, save_fig
from src.2D1PopNeuronModel import plot_kernel, plot_influence, plot_angular_variance, plot_variance_comparison, plot_variance_comparison_diff
from src.2D1PopNeuronModel import plot_radius_variance_comparison, plot_variance_comparison_overlay, set_first_last_ticksize, set_imshow_ticks
from src.2D1PopNeuronModel import plot_radius_variance_all_locations, fftind, gaussian_random_field, Func, plot_variance_comparison_across_locations
from src.2D1PopNeuronModel import plot_variance_diff_across_locations, zero_if_small

In [ ]:
export_dir = '/path/for/plots'
os.makedirs(export_dir, exist_ok=True)

In [ ]:
# Network Configuration

# Cross-Dominant
N = 50
a = 3.0
sigma_e = 4
r = 1.5

In [ ]:
# Not Cross-Dominant
N = 50
a = 0.75
sigma_e = 4
r = 2.5

In [ ]:
# Gaussian Random Field

F1GT, _ = Func(N, seed=1, alpha_grf = 0)
field = np.reshape(F1GT.T[1275], (N, N))
fig, ax = plt.subplots(figsize=(6, 6), dpi=120)  
vmax = np.max(np.abs(field))
im = ax.imshow(
    field,
    cmap='bwr',   
    vmin=-vmax,
    vmax=vmax,
)
ax.set_title("GRF-based Modulation Field\nNeuron 1275", fontsize=12)
ax.set_xlabel("x", fontsize=13)
ax.set_ylabel("y", fontsize=13)
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(2)
    spine.set_color('black')
cbar = plt.colorbar(im, ax=ax, shrink=0.5)
cbar.set_label("Field value", rotation=270, labelpad=15)
plt.tight_layout()
plt.savefig("GRF.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# Connectivity and Influence Kernels

center_col = N // 2 * N + N // 2
center = 25 * N + 25        # (25, 25)
top_left = 3 * N + 3        # (3, 3)
bottom_right = 47 * N + 47  # (47, 47)
stimulus_locations = [center, top_left, bottom_right]

# 1. Create Kernels
M0 = recurrent_connections_2d(N, a, sigma_e, r)
P0 = M0.copy()

# 2. Perturb 
Mp1 = M0 + 1.5 * P0 * F1GT
Mp2 = M0 + 0.5 * P0 * F1GT
Mp3 = M0 + 2.5 * P0 * F1GT

# 3. Compute influences 
stim_strength = 1.0
influence_M0 = compute_influence_2d(M0, stimulus_locations, stim_strength, N)
influence_Mp1 = compute_influence_2d(Mp1, stimulus_locations, stim_strength, N)
influence_Mp2 = compute_influence_2d(Mp2, stimulus_locations, stim_strength, N)
influence_Mp3 = compute_influence_2d(Mp3, stimulus_locations, stim_strength, N)

# 4. Plot 
plot_kernel(M0, stimulus_locations, N, 'Connectivity', filename='M0_connectivity' )
plot_influence(influence_M0, stimulus_locations, N, 'Influence', filename='M0_influence')

plot_kernel(Mp2, stimulus_locations, N, 'Connectivity',filename='Mp2_connectivity')
plot_influence(influence_Mp2, stimulus_locations, N, 'Influence',filename='Mp2_influence')

plot_kernel(Mp1, stimulus_locations, N, 'Connectivity',filename='Mp1_connectivity')
plot_influence(influence_Mp1, stimulus_locations, N, 'Influence',filename='Mp1_influence')

plot_kernel(Mp3, stimulus_locations, N, 'Connectivity', filename='Mp3_connectivity')
plot_influence(influence_Mp3, stimulus_locations, N, 'Influence',filename='Mp3_influence')

In [ ]:
# Homogeneous Kernel - Mean values over 360 angles
kernel_col_M0  = center_kernel(M0.T[1275].reshape(N, N),  1275, N)
_, _, slices = compute_angular_variance_2d(kernel_col_M0, N)

mean_per_radius = np.mean(slices, axis=0)  
r = np.arange(len(mean_per_radius))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(r, mean_per_radius, color='steelblue', linewidth=2)
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xlabel('Radius $r$', fontsize=12)
ax.set_ylabel('Mean value over 360 angles', fontsize=12)
ax.set_title('$\\bar{s}(r)$ - Homogeneous Kernel', fontsize=13, fontweight='bold')
ax.set_xlim(0, N//2-1)
for spine in ax.spines.values():
    spine.set_linewidth(2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("mean_s_r_hom.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# Heterogeneous Kernel - Mean values over 360 angles
kernel_col_Mp3  = center_kernel(Mp3.T[1275].reshape(N, N),  1275, N)
_, _, slices = compute_angular_variance_2d(kernel_col_Mp3, N)

mean_per_radius = np.mean(slices, axis=0)  
r = np.arange(len(mean_per_radius))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(r, mean_per_radius, color='steelblue', linewidth=2)
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xlabel('Radius $r$', fontsize=12)
ax.set_ylabel('Mean value over 360 angles', fontsize=12)
ax.set_title('$\\bar{s}(r)$ - Perturbed Kernel', fontsize=13, fontweight='bold')
ax.set_xlim(0, N//2-1)
for spine in ax.spines.values():
    spine.set_linewidth(2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("mean_s_r_per.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# Angular Slices - Homogeneous vs. Heterogeneous
kernel_col_Mp0 = center_kernel(M0.T[1275].reshape(N, N), 1275, N)
_, _, slices_0 = compute_angular_variance_2d(kernel_col_Mp0, N)
kernel_col_Mp3 = center_kernel(Mp3.T[1275].reshape(N, N), 1275, N)
_, _, slices = compute_angular_variance_2d(kernel_col_Mp3, N)

r = np.arange(slices.shape[1])
n_angles = slices.shape[0]
angle_indices = {
    '0°':   0,
    '90°':  n_angles // 4,
    '180°': n_angles // 2,
    '270°': 3 * n_angles // 4,
}

colors = ['steelblue', 'tomato', 'seagreen', 'darkorange']

fig, ax = plt.subplots(figsize=(9, 5))

for (label, idx), color in zip(angle_indices.items(), colors):

    ax.plot(r, slices[idx],   color=color, linewidth=2,   linestyle='-',  label=f'{label} perturbed')
    ax.plot(r, slices_0[idx], color=color, linewidth=1.5, linestyle='--', label=f'{label} unperturbed', alpha=0.7)

ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_xlabel('Radius $r$', fontsize=12)
ax.set_ylabel('Connectivity', fontsize=12)
ax.set_title('Angular Slices — Perturbed vs. Unperturbed Kernel', fontsize=13, fontweight='bold')
ax.set_xlim(0, N//2 - 1)


ax.legend(fontsize=10, framealpha=0.8, ncol=2)

for spine in ax.spines.values():
    spine.set_linewidth(2)

ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("slices_angular_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# Variance Analysis

center_col    = N // 2 * N + N // 2
top_left_col  = 3 * N + 3
bot_right_col = 47 * N + 47

col_stim_pairs = [
    (center_col,    stimulus_locations[0]),
    (top_left_col,  stimulus_locations[1]),
    (bot_right_col, stimulus_locations[2])
]

models      = [M0,  Mp2, Mp1,   Mp3]
inf_models  = [influence_M0, influence_Mp2, influence_Mp1, influence_Mp3]
labels = ['No Perturbation', 'Low (β=0.1)', 'Moderate (β=1.5)', 'High (β=2.5)']

results = {}  

for col, stim_loc in col_stim_pairs:
    results[(col, stim_loc)] = []

    for M, inf_M in zip(models, inf_models):

        # Connectivity
        kernel = center_kernel(M.T[col].reshape(N, N), col, N)
        var_conn, mean_conn, slices_conn, var_conn_slice = compute_angular_variance_2d(kernel, N)
        var_conn = uncenter_field(var_conn, col, N)

        # Influence
        inf_2d = center_kernel(inf_M[stim_loc].reshape(N, N), stim_loc, N)
        var_inf, mean_inf, slices_inf, var_inf_slice = compute_angular_variance_2d(inf_2d, N)
        var_inf = uncenter_field(var_inf, stim_loc, N)

        # Radius variance
        var_r_conn = np.var(slices_conn, axis=0)
        var_r_inf  = np.var(slices_inf,  axis=0)

        results[(col, stim_loc)].append({
            'var_conn': var_conn, 'mean_conn': mean_conn,
            'slices_conn': slices_conn, 'var_conn_slice': var_conn_slice,
            'var_inf':  var_inf,  'mean_inf':  mean_inf,
            'slices_inf':  slices_inf,  'var_inf_slice':  var_inf_slice,
            'var_r_conn': var_r_conn,
            'var_r_inf':  var_r_inf,
        })

FORCE_ZERO_KEYS = {'var_conn', 'var_inf', 'var_r_conn', 'var_r_inf'}

for col, stim_loc in col_stim_pairs:
    for i, r in enumerate(results[(col, stim_loc)]):
        for key in FORCE_ZERO_KEYS:
            r[key] = zero_if_small(r[key])
        if i == 0:  # M0
            for key in FORCE_ZERO_KEYS:
                r[key] = r[key].copy()
                r[key][:] = 0

# norm everyhting > pert strengths + locations
all_var_fields  = [r[k] for (col, stim_loc) in col_stim_pairs
                        for r in results[(col, stim_loc)]
                        for k in ('var_inf', 'var_conn')]

all_radius_vals = [r[k] for (col, stim_loc) in col_stim_pairs
                        for r in results[(col, stim_loc)]
                        for k in ('var_r_inf', 'var_r_conn')]

global_vmax        = max(v.max() for v in all_var_fields)
global_radius_vmax = max(v.max() for v in all_radius_vals)
global_norm        = mpl.colors.Normalize(vmin=0, vmax=global_vmax)

loc_names = ['Center', 'Top-Left', 'Bottom-Right']

radius_conn_per_loc = {}
radius_inf_per_loc  = {}

for (col, stim_loc), loc_name in zip(col_stim_pairs, loc_names):
    rs = results[(col, stim_loc)]
    radius_conn_per_loc[loc_name] = [r['var_r_conn'] for r in rs]
    radius_inf_per_loc[loc_name]  = [r['var_r_inf']  for r in rs]

plot_radius_variance_all_locations(
    var_curves_per_location=radius_conn_per_loc,
    pert_labels=labels,
    suptitle='Variance per Radius (Connectivity) – All Locations',
    filename='radius_variance_conn_all_locs.png',
    shared_ymax=global_radius_vmax
)
plot_radius_variance_all_locations(
    var_curves_per_location=radius_inf_per_loc,
    pert_labels=labels,
    suptitle='Variance per Radius (Influence) – All Locations',
    filename='radius_variance_inf_all_locs.png',
    shared_ymax=global_radius_vmax
)

for (col, stim_loc), loc_name in zip(col_stim_pairs, loc_names):
    rs = results[(col, stim_loc)]

    local_vmax = max(r['var_inf'].max() for r in rs[1:])
    local_norm = mpl.colors.Normalize(vmin=0, vmax=local_vmax)

    norm = global_norm  

    var_conn_list  = [r['var_conn']    for r in rs]
    var_inf_list   = [r['var_inf']     for r in rs]

    pert_titles = [
        'No Perturbation', 'Low Perturbation', 'Moderate Perturbation', 'High Perturbation'
    ]

    for r, pt, lbl in zip(rs, pert_titles, labels):
        plot_angular_variance(
            r['var_conn'], r['mean_conn'], r['slices_conn'], r['var_conn_slice'], N,
            title=f'Connectivity Variance Map – {pt} | {loc_name}',
            col_loc=col, norm=norm,
            filename=f'conn_var_{lbl.split()[0]}_{loc_name}.png'
        )

    for r, pt, lbl in zip(rs, pert_titles, labels):
        plot_angular_variance(
            r['var_inf'], r['mean_inf'], r['slices_inf'], r['var_inf_slice'], N,
            title=f'Influence Variance Map – {pt} | {loc_name}',
            stim_loc=stim_loc, norm=norm,
            filename=f'inf_var_{lbl.split()[0]}_{loc_name}.png'
        )

    plot_variance_comparison(
        var_fields=var_conn_list, labels=labels, N=N, norm=norm,
        suptitle=f'Connectivity Variance Vergleich | {loc_name}',
        filename=f'conn_var_comparison_{loc_name}.png'
    )
    plot_variance_comparison(
        var_fields=var_inf_list, labels=labels, N=N, norm=norm,
        suptitle=f'Influence Variance Vergleich | {loc_name}',
        stim_loc=stim_loc,
        filename=f'inf_var_comparison_{loc_name}.png'
    )

    var_diff_list = [vi - vc for vi, vc in zip(var_inf_list, var_conn_list)]
    plot_variance_comparison_diff(
        var_fields=var_diff_list, labels=labels, N=N, norm=norm,
        suptitle=f'Differenz Influence–Connectivity | {loc_name}',
        stim_loc=stim_loc,
        filename=f'variance_difference_{loc_name}.png'
    )
    
pert_titles = ['No Perturbation', 'Low (β=0.1)', 'Moderate (β=1.5)', 'High (β=2.5)']

for pert_idx, pert_label in enumerate(pert_titles):
    if pert_idx == 0:  
        continue
    lbl = pert_label.split()[0]

    plot_variance_comparison_across_locations(
        results=results, col_stim_pairs=col_stim_pairs, loc_names=loc_names,
        pert_idx=pert_idx, pert_label=pert_label, N=N, norm=global_norm,
        key='var_inf', filename=f'inf_var_locs_{lbl}.png'
    )
    plot_variance_comparison_across_locations(
        results=results, col_stim_pairs=col_stim_pairs, loc_names=loc_names,
        pert_idx=pert_idx, pert_label=pert_label, N=N, norm=global_norm,
        key='var_conn', filename=f'conn_var_locs_{lbl}.png'
    )
    plot_variance_diff_across_locations(
        results=results, col_stim_pairs=col_stim_pairs, loc_names=loc_names,
        pert_idx=pert_idx, pert_label=pert_label, N=N,
        filename=f'diff_var_locs_{lbl}.png'
    )